# ETL Transform: News

This notebook runs the **news ETL pipeline**: ingest from Postgres → transform → save to `financial_news_transformed` → publish to S3.

**Two modes (set `USE_AGENTIC_ONLY` in the next cell):**
- **VADER (default)**: transform with sentiment (VADER), intent, keywords, tickers; then save and S3.
- **Agentic only**: skip VADER; run LLM-based financial metrics extraction only; then save and S3.

**S3 upload modes:**
- **Per-article**: one CSV per article at `news/crypto/[agentic=true|false/]year=.../.../format=csv/{id}.csv`
- **Batch (run)**: one CSV per run at `news/transformed/crypto/.../batch=run/...` (by run time)
- **Batch (year/month/week/day)**: partitioned by **article issue date** under `news/transformed/crypto/year=.../month=.../day=...` etc.

Set `AWS_NEWS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads. For agentic mode, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` and optionally `LLM_PROVIDER` (default: openai).

In [1]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [ ]:
# Options: set to True to skip VADER and use only agentic (LLM) enrichment
USE_AGENTIC_ONLY = False
SINCE = "2025-01-01"
UNTIL = "2025-01-31"
# For agentic-only: limit rows (None = no limit; set e.g. 10 for a quick test)
AGENTIC_MAX_ROWS = None
# S3: per-article and/or batch (same for both modes)
UPLOAD_S3_PER_ARTICLE = True
UPLOAD_S3_BATCH = ["run", "week", "month", "year", "day"]

In [3]:
# Run news ETL: either agentic-only (skip VADER) or VADER transform; then save to Postgres and S3.
# Agentic: only saves to DB/S3 when enrichment succeeds (no errors). Install: pip install openai

from datetime import datetime
from pipelines.etl_transform import (
    run_news_etl,
    save_transformed_news_to_postgres,
    _ensure_news_transformed_table,
    agentic_result_has_failures,
    build_s3_key_news_per_article,
    upload_news_batches_to_s3,
    upload_dataframe_to_s3_key,
)
from config.settings import get_settings
from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries as q

if USE_AGENTIC_ONLY:
    from pipelines.etl_cli import ingest_news
    from agents.registry import get_llm_client
    from agents.transforms.agentic_transform import AgenticTextEnricher, FinancialMetricsTask
    from storage.cloud.CloudStorage import CloudStorageProvider

    df = ingest_news(since=SINCE, until=UNTIL)
    if df is None or df.empty:
        transformed_df = pd.DataFrame()
        print("No news data to process")
    else:
        settings = get_settings()
        provider = getattr(settings.agent, "provider", "openai")
        transformed_df = pd.DataFrame()
        try:
            client = get_llm_client(provider)
            enricher = AgenticTextEnricher(client=client, task=FinancialMetricsTask())
            transformed_df = enricher.enrich_dataframe(df, max_rows=AGENTIC_MAX_ROWS)
        except (KeyError, ValueError) as e:
            print(f"Agentic skipped (LLM not configured): {e}")
        except Exception as e:
            print(f"Agentic failed (no data saved to DB or S3): {e}")
            transformed_df = pd.DataFrame()

        # Only save to Postgres and S3 when enrichment succeeded (no per-row llm_error)
        if not transformed_df.empty and not agentic_result_has_failures(transformed_df):
            conn = PgConn(q.FINANCIAL_NEWS_TABLE_NAME)
            _ensure_news_transformed_table(conn)
            n = save_transformed_news_to_postgres(conn, transformed_df, agentic_enabled=True)
            conn.close_connection()
            print(f"Saved {n} rows to {q.FINANCIAL_NEWS_TRANSFORMED_TABLE_NAME}")

            bucket = getattr(settings.aws, "news_bucket", None) or settings.aws.default_bucket
            if bucket and (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH):
                aws = CloudStorageProvider.AWS()
                if UPLOAD_S3_PER_ARTICLE:
                    for _, row in transformed_df.iterrows():
                        aid = str(row.get("id", ""))
                        dt_str = row.get("datetime")
                        try:
                            dt = pd.to_datetime(dt_str) if dt_str else datetime.utcnow()
                        except Exception:
                            dt = datetime.utcnow()
                        key = build_s3_key_news_per_article(aid, dt, agentic=True)
                        upload_dataframe_to_s3_key(aws.s3_client, bucket, key, pd.DataFrame([row]))
                    print(f"Uploaded {len(transformed_df)} per-article CSVs to s3://{bucket}/")
                if UPLOAD_S3_BATCH:
                    upload_news_batches_to_s3(aws.s3_client, bucket, transformed_df, UPLOAD_S3_BATCH, agentic=True)
                    print(f"Uploaded batches {UPLOAD_S3_BATCH} to s3://{bucket}/")
        elif not transformed_df.empty and agentic_result_has_failures(transformed_df):
            print("Agentic enrichment had errors (llm_error set on one or more rows). Not saving to DB or S3.")

    print(f"Agentic: transformed {len(transformed_df)} articles")
else:
    transformed_df = run_news_etl(
        since=SINCE,
        until=UNTIL,
        news_bucket=None,
        save_to_postgres=True,
        upload_s3_per_article=UPLOAD_S3_PER_ARTICLE,
        upload_s3_batch=UPLOAD_S3_BATCH if UPLOAD_S3_BATCH else None,
        sentiment_backend="vader",
        extract_tickers=True,
    )
    print(f"VADER: transformed {len(transformed_df)} articles")

Connection to the database successful!
Table name set to: financial_news_241118
Connection closed.
Connection to the database successful!
Table name set to: financial_news_241118
Table name set to: financial_news_transformed
Table 'financial_news_transformed' already exists.
Table name set to: financial_news_transformed
Saving to postgres db...done
Saving to postgres db...done
Saving to postgres db...done
Saving to postgres db...done
Saving to postgres db...done
Saving to postgres db...done
Saving to postgres db...done
Saving to postgres db...done
Saving to postgres db...done
Connection closed.
Saved 9 rows to financial_news_transformed
Uploaded 9 per-article CSVs to s3://test-financial-news-bucket/
Uploaded batch run to s3://test-financial-news-bucket/news/transformed/crypto/agentic=true/batch=run/format=csv/news_transformed_20260305_025341.csv
Uploaded batch week to s3://test-financial-news-bucket/news/transformed/crypto/agentic=true/year=2026/week=10/format=csv/news_transformed_y202

In [4]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

,id,source,headline,href,summary,content,author,minsread,datetime,llm_financial_metrics,...,llm_impact_strength,llm_immediacy,llm_impact_horizon,llm_confidence,llm_sentiment_label,llm_impact_level,llm_signal,llm_actionable,llm_sectors,llm_key_facts
301,186493912166739695299362559442695803429,Motley Fool,3 Predictions for Crypto in 2025,https://finance.yahoo.com/news/3-predictions-c...,"Bitcoin (CRYPTO: BTC) broke the $100,000 barri...",Last year was remarkable for cryptocurrency. B...,"RJ Fulton, The Motley Fool",5 min read,2025-01-01T10:30:00.000Z,"{'ticker': 'BTC', 'event_type': 'other', 'over...",...,0.9,0.6,medium_term,0.8,positive,high,bullish,True,[crypto],"[Bitcoin is predicted to reach $200,000 in 202..."
309,270218805936899518274975867363475896779,Motley Fool,3 Reasons Bitcoin Is a Must-Buy for Long-Term ...,https://finance.yahoo.com/news/3-reasons-bitco...,Bitcoin (CRYPTO: BTC) has had a phenomenal yea...,Bitcoin (CRYPTO: BTC) has had a phenomenal yea...,"Neil Patel, The Motley Fool",4 min read,2025-01-01T14:17:00.000Z,"{'ticker': 'BTC', 'event_type': 'other', 'over...",...,0.8,0.7,long_term,0.9,positive,high,bullish,True,[crypto],"[Bitcoin has surged 122% in 2024., Bitcoin's m..."
1853,186493912166739695299362559442695803429,Bloomberg,Bitcoin Slips in December as Investors Cash In...,https://finance.yahoo.com/news/bitcoin-slips-d...,(Bloomberg) -- Bitcoin’s record-breaking run f...,(Bloomberg) -- Bitcoin’s record-breaking run f...,Sidhartha Shukla,1 min read,2025-01-01T13:26:06.000Z,"{'ticker': 'BTC', 'event_type': 'other', 'over...",...,0.6,0.7,short_term,0.8,negative,medium,bearish,True,[crypto],"[Bitcoin fell 3.2% last month, Bitcoin reached..."
3832,270218805936899518274975867363475896779,CoinDesk,Crypto for Advisors: What’s Next for Crypto ET...,https://finance.yahoo.com/news/crypto-advisors...,Crypto for Advisors: What’s Next for Crypto ET...,"Happy New Year, advisors! We look forward to b...","Roxanna Islam, Sarah Morton",6 min read,2025-01-01T19:00:10.000Z,"{'ticker': '', 'event_type': 'other', 'overall...",...,0.7,0.5,medium_term,0.8,positive,high,bullish,True,"[crypto, equities]",[ETFs brought in over $1 trillion in net inflo...
5448,270218805936899518274975867363475896779,CoinMarketCap,IRS Delays Implementation of New Crypto Cost-B...,https://finance.yahoo.com/news/irs-delays-impl...,The Internal Revenue Service (IRS) has announc...,IRS Delays Implementation of New Crypto Cost-B...,Decentralized Dog,1 min read,2025-01-01T17:31:00.000Z,"{'ticker': '', 'event_type': 'regulation', 'ov...",...,0.4,0.5,medium_term,0.7,neutral,medium,neutral,True,"[crypto, regulation]",[IRS delays implementation of new crypto cost-...


['id', 'source', 'headline', 'href', 'summary', 'content', 'author', 'minsread', 'datetime', 'llm_financial_metrics', 'llm_entities', 'llm_ticker', 'llm_event_type', 'llm_overall_sentiment', 'llm_forward_sentiment', 'llm_surprise_score', 'llm_risk_score', 'llm_uncertainty_score', 'llm_impact_strength', 'llm_immediacy', 'llm_impact_horizon', 'llm_confidence', 'llm_sentiment_label', 'llm_impact_level', 'llm_signal', 'llm_actionable', 'llm_sectors', 'llm_key_facts']


## Optional: Per-article only (no batch)
Use when you want only one CSV per article in S3.

In [5]:
# transformed_per_article = run_news_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     save_to_postgres=True,
#     upload_s3_per_article=True,
#     upload_s3_batch=None,
# )

## Optional: Batch only (no per-article)
Use when you want a single CSV per run, or per week/month/year.

In [ ]:
# transformed_batch = run_news_etl(
#     date="2026-01-27",
#     save_to_postgres=True,
#     upload_s3_per_article=False,
#     upload_s3_batch=["run", "month"],
# )